# P1-3 · Cây quyết định và ensemble

Toàn bộ code của bài, chạy từ trên xuống như trên trang bài học. Kết quả đã chạy sẵn được giữ trong notebook; bấm **Runtime → Run all** để chạy lại. Ô có dòng đầu `# Cần GPU` thì đổi **Runtime → Change runtime type → T4 GPU**; ô cần API key thì đặt khóa trong Colab Secrets.

In [ ]:
!pip install -q lightgbm

## Lý thuyết

In [ ]:
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text

# 10 khách hàng của ví dụ trên: cây độ sâu 1 phải chọn đúng ngưỡng 27.5
X10 = pd.DataFrame({"thu_nhap": [8, 12, 15, 20, 25, 30, 35, 40, 50, 60]})
y10 = [0, 0, 0, 1, 0, 1, 1, 1, 1, 1]
stump = DecisionTreeClassifier(max_depth=1).fit(X10, y10)
print(export_text(stump, feature_names=["thu_nhap"]))

# 569 khối u, 30 đặc trưng số, nhãn lành/ác (có sẵn trong scikit-learn, không cần tải)
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
tree = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X_train, y_train)
print(export_text(tree, feature_names=list(X_train.columns)))   # in cây dạng chữ

|--- thu_nhap <= 27.50
|   |--- class: 0
|--- thu_nhap >  27.50
|   |--- class: 1

|--- worst perimeter <= 106.10
|   |--- worst concave points <= 0.13
|   |   |--- area error <= 48.98
|   |   |   |--- class: 1
|   |   |--- area error >  48.98
|   |   |   |--- class: 0
|   |--- worst concave points >  0.13
|   |   |--- worst texture <= 26.00
|   |   |   |--- class: 1
|   |   |--- worst texture >  26.00
|   |   |   |--- class: 0
|--- worst perimeter >  106.10
|   |--- worst concave points <= 0.14
|   |   |--- worst radius <= 17.61
|   |   |   |--- class: 1
|   |   |--- worst radius >  17.61
|   |   |   |--- class: 0
|   |--- worst concave points >  0.14
|   |   |--- texture error <= 0.43
|   |   |   |--- class: 1
|   |   |--- texture error >  0.43
|   |   |   |--- class: 0



In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=500, min_samples_leaf=2, n_jobs=-1, oob_score=True, random_state=0)
rf.fit(X_train, y_train)
rf.oob_score_          # ước lượng accuracy không cần tập validation riêng

0.9624413145539906


In [ ]:
import lightgbm as lgb
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.25, stratify=y_train, random_state=0)
model = lgb.LGBMClassifier(n_estimators=5000, learning_rate=0.03, num_leaves=31, verbose=-1)
model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)],
          callbacks=[lgb.early_stopping(100)])     # dừng khi 100 cây liền không cải thiện val
model.best_iteration_                              # số cây thật sự dùng

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[122]	valid_0's binary_logloss: 0.134858
122


In [ ]:
from sklearn.inspection import permutation_importance
r = permutation_importance(model, X_val, y_val, n_repeats=10, random_state=0)
sorted(zip(r.importances_mean, X_val.columns), reverse=True)[:10]

[(0.03271028037383178, 'worst radius'), (0.012149532710280376, 'worst texture'), (0.008411214953271028, 'worst area'), (0.0009345794392523366, 'worst perimeter'), (0.0, 'worst symmetry'), (0.0, 'worst smoothness'), (0.0, 'worst fractal dimension'), (0.0, 'worst concavity'), (0.0, 'worst compactness'), (0.0, 'texture error')]


## Bài tập 3: Đoán trước rồi kiểm chứng {khái niệm}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
X, y = make_classification(2000, 20, n_informative=6, flip_y=0.1, random_state=0)
Xtr, Xva, ytr, yva = train_test_split(X, y, random_state=0)
for d in [1, 3, 5, 8, 12, 20]:
    t = DecisionTreeClassifier(max_depth=d, random_state=0).fit(Xtr, ytr)
    print(d, round(t.score(Xtr, ytr), 3), round(t.score(Xva, yva), 3))
rf = RandomForestClassifier(300, max_depth=20, n_jobs=-1, random_state=0).fit(Xtr, ytr)
print("RF", rf.score(Xva, yva))

1 0.761 0.754
3 0.836 0.796
5 0.904 0.828
8 0.965 0.81
12 0.996 0.806
20 1.0 0.8
RF 0.868


## Bài tập 4: LightGBM với early stopping {code}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import lightgbm as lgb, matplotlib.pyplot as plt, pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance

data = load_breast_cancer(as_frame=True)
Xtr, Xva, ytr, yva = train_test_split(data.data, data.target, test_size=0.25, stratify=data.target, random_state=0)
m = lgb.LGBMClassifier(n_estimators=2000, learning_rate=0.05, num_leaves=15, verbose=-1)
m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
print("số cây tốt nhất:", m.best_iteration_, "| acc val:", m.score(Xva, yva))
pi = permutation_importance(m, Xva, yva, n_repeats=10, random_state=0)
pd.Series(pi.importances_mean, index=Xtr.columns).nlargest(10).plot.barh(); plt.show()

số cây tốt nhất: 89 | acc val: 0.9300699300699301


## Bài tập 5: Cây quyết định duyệt khoản vay {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
n = 1000
income = rng.lognormal(2.7, 0.5, n)                 # triệu đồng/tháng
debt_ratio = rng.uniform(0, 1, n)
late = rng.poisson(0.6, n)
risk = 1.5 * debt_ratio + 0.6 * late - 0.03 * income + rng.normal(0, 0.3, n)
default = (risk > 0.9).astype(int)                  # 1 = vỡ nợ
X = np.column_stack([income, debt_ratio, late]); ten_cot = ["income", "debt_ratio", "late"]
# TODO: chia train/test (random_state=0), fit cây depth 2, in luật, acc, quyet_dinh cho người mới

In [ ]:
X_tr, X_te, y_tr, y_te = train_test_split(X, default, test_size=0.25, random_state=0)
cay = DecisionTreeClassifier(max_depth=2, random_state=0).fit(X_tr, y_tr)
print(export_text(cay, feature_names=ten_cot))
acc = cay.score(X_te, y_te)
quyet_dinh = cay.predict([[18, 0.55, 0]])[0]
print(f"accuracy test = {acc:.3f}; người mới: {'từ chối' if quyet_dinh else 'duyệt'}")

|--- late <= 0.50
|   |--- debt_ratio <= 0.77
|   |   |--- class: 0
|   |--- debt_ratio >  0.77
|   |   |--- class: 0
|--- late >  0.50
|   |--- debt_ratio <= 0.57
|   |   |--- class: 0
|   |--- debt_ratio >  0.57
|   |   |--- class: 1

accuracy test = 0.808; người mới: duyệt


## Bài tập 6: Tìm độ sâu phù hợp {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
# TODO: acc_tr, acc_va cho depth 1..15; sau_tot; acc_leaf

In [ ]:
depths = range(1, 16)
acc_tr, acc_va = [], []
for d in depths:
    c = DecisionTreeClassifier(max_depth=d, random_state=0).fit(X_tr, y_tr)
    acc_tr.append(c.score(X_tr, y_tr)); acc_va.append(c.score(X_va, y_va))
sau_tot = int(np.argmax(acc_va) + 1)
acc_leaf = DecisionTreeClassifier(min_samples_leaf=10, random_state=0).fit(X_tr, y_tr).score(X_va, y_va)

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.plot(depths, acc_tr, marker="o", label="train"); ax.plot(depths, acc_va, marker="o", label="validation")
ax.axvline(sau_tot, color="gray", ls="--"); ax.legend()
ax.set(xlabel="max_depth", ylabel="accuracy", title="Cây sâu thuộc lòng train, validation không khá hơn")
print(f"độ sâu tốt nhất: {sau_tot}, validation {max(acc_va):.3f}; min_samples_leaf=10: {acc_leaf:.3f}")

độ sâu tốt nhất: 3, validation 0.953; min_samples_leaf=10: 0.936


## Bài tập 7: Một cây hay cả khu rừng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.datasets import load_breast_cancer
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
X, y = load_breast_cancer(return_X_y=True)
# TODO: acc_cay, acc_rung (mỗi list 10 số); oob

In [ ]:
acc_cay, acc_rung = [], []
for s in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=s, stratify=y)
    acc_cay.append(DecisionTreeClassifier(random_state=0).fit(X_tr, y_tr).score(X_te, y_te))
    rung = RandomForestClassifier(n_estimators=200, oob_score=True, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    acc_rung.append(rung.score(X_te, y_te))
oob = rung.oob_score_
print(f"một cây: {np.mean(acc_cay):.3f} ± {np.std(acc_cay):.3f}")
print(f"forest : {np.mean(acc_rung):.3f} ± {np.std(acc_rung):.3f}   (OOB lần chia cuối: {oob:.3f})")

một cây: 0.930 ± 0.025
forest : 0.961 ± 0.010   (OOB lần chia cuối: 0.967)


## Bài tập 8: Đặc trưng nào thật sự quan trọng {ứng dụng}

(Tự làm trước trên trang bài học; các ô dưới có cả lời giải.)

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance
from sklearn.model_selection import train_test_split
rng = np.random.default_rng(0)
n = 2000
thu_nhap = rng.normal(0, 1, n)
thu_nhap_usd = thu_nhap + rng.normal(0, 0.05, n)
tuoi = rng.normal(0, 1, n)
ma_kh = rng.permutation(n).astype(float)
y = (1.5 * thu_nhap + 0.5 * tuoi + rng.normal(0, 0.8, n) > 0).astype(int)
X = np.column_stack([thu_nhap, thu_nhap_usd, tuoi, ma_kh])
ten = ["thu_nhap", "thu_nhap_usd", "tuoi", "ma_kh"]

def do_importance(Xc):
    X_tr, X_va, y_tr, y_va = train_test_split(Xc, y, test_size=0.3, random_state=0)
    rf = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=0, n_jobs=-1).fit(X_tr, y_tr)
    perm = permutation_importance(rf, X_va, y_va, n_repeats=10, random_state=0).importances_mean
    return rf.feature_importances_, perm
# TODO: gini, perm = do_importance(X); gini2, perm2 = do_importance(X bỏ cột 1)

In [ ]:
gini, perm = do_importance(X)
gini2, perm2 = do_importance(X[:, [0, 2, 3]])
for i, t in enumerate(ten):
    print(f"{t:13s} Gini {gini[i]:.3f}   permutation {perm[i]:+.3f}")
print("bỏ thu_nhap_usd → thu_nhap: Gini", round(gini2[0], 3), " permutation", round(perm2[0], 3))

thu_nhap      Gini 0.395   permutation +0.121
thu_nhap_usd  Gini 0.410   permutation +0.188
tuoi          Gini 0.141   permutation +0.046
ma_kh         Gini 0.054   permutation -0.004
bỏ thu_nhap_usd → thu_nhap: Gini 0.759  permutation 0.353
